# Imports

In [1]:
import numpy as np
import pandas as pd
import polars as pl
import regex as re_adv
import contractions, torch, swifter
import modin.pandas as md
import ray
from sentence_transformers import SentenceTransformer
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
import warnings
warnings.filterwarnings("ignore")

Device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Data Imports

In [ ]:
df = pd.read_json("data.ndjson", lines=True)
df.head(5)

## Data Processing

In [ ]:
df.columns

In [ ]:
df = df[["name","type","language","genres","premiered","ended","rating","summary"]]

## Data Analysis

In [ ]:
df = df.dropna(subset=["summary"], axis="rows")
df.head()

In [ ]:
df.info(memory_usage="deep")

In [ ]:
df.genres.value_counts()

The genres column might look having no missing data, but it has about 2541 missing datas, we assume these as "Entertainment" category since these data cannot be recovered back and assuming mean or median can be useless.

In [ ]:
df.rating.value_counts()

Ratings stored in wrong format, and most of the data is missing, which is way beyond recovery or imputation, so we keep those as `unknown` or `no data`, so our LLM doesn't gets confused of it.<br>
The summary column needs further text processing too

## Data Cleaning

In [ ]:
df['language'] = df['language'].fillna('English')

df.genres = df.genres.astype("str")

In [ ]:
df['genres'] = df['genres'].replace("[]", "Entertainment")

df['genres'] = df['genres'].str.replace(r"[\[\]\']", "", regex=True)


In [ ]:
df.genres.value_counts().head() #Fixed

In [ ]:
df.rating = df.rating.astype("str")
df.rating = df.rating.str[12:15].replace("}",".0").replace("Non","no data")

In [ ]:
df.premiered = df.premiered.fillna("unknown").astype(str)
df.ended = df.ended.fillna("unknown").astype(str)

In [ ]:
df.info(memory_usage="deep")

In [ ]:
df.head()

Our data is cleaned and set for the next phase.

## Multiplying data for simulating bigger dataset processing

In [ ]:
df.shape

In [ ]:
for _ in range(9):
    df = pd.concat([df,df], ignore_index=True)

In [ ]:
df.shape

In [ ]:
df.info(memory_usage="deep")

In [ ]:
df.to_parquet("big_data.parquet", index=False)
del df

We multiplied your data from 9MB to 4.5GB, now we will see how the Pandas, Modin Pandas and Polar works on this big dataset.

## Data Loading Test

In [2]:
import pandas as pd
import ray
# Initialize Ray here so it starts immediately
ray.init(num_cpus=4) 
import modin.pandas as md



2026-04-09 15:38:30,570	INFO worker.py:2013 -- Started a local Ray instance.


In [3]:
%time df = pd.read_parquet("big_data.parquet")
del df

CPU times: total: 11.1 s
Wall time: 20.2 s


In [4]:
%time df = md.read_parquet("big_data.parquet")
del df

CPU times: total: 672 ms
Wall time: 20.7 s


In [5]:
import polars as pl
%time df_pl = pl.read_parquet("big_data.parquet")
del df_pl

CPU times: total: 4.48 s
Wall time: 4.53 s


## Processing Performance Comparisons (Big Data)
Now we will load the big data and compare the performance of standard Pandas, Swifter, Modin, and Polars for text processing.

In [14]:
# Reloading the big data for processing comparison
df = pd.read_parquet("big_data.parquet")
print(f"Data loaded with shape: {df.shape}")

Data loaded with shape: (5163008, 8)


## Vectorization

In [3]:
# 1. Specific Bold Tag Handlers
BOLD_OPEN_PATTERN = re_adv.compile(r'<b>')
BOLD_CLOSE_PATTERN = re_adv.compile(r'\s*</b>')

# 2. General HTML/Entity Handlers (to catch everything else like <p>)
HTML_PATTERN = re_adv.compile(r'<.*?>')
ENTITY_PATTERN = re_adv.compile(r'&\w+;')

# 3. Updated Clean Pattern (Now allowing " and ' as discussed)
CLEAN_PATTERN = re_adv.compile(r'[^\p{L}\p{N}\s.,!?"\']', flags=re_adv.V1)

# 4. Whitespace and Dash
DASH_PATTERN = re_adv.compile(r'-')
WHITESPACE_PATTERN = re_adv.compile(r'\s+')

def text_process(text):
    if not isinstance(text, str): return ""
    
    # --- CONVERSION STEP ---
    # Convert <b>...</b> to "..."
    text = BOLD_OPEN_PATTERN.sub('"', text)
    text = BOLD_CLOSE_PATTERN.sub('" ', text)
    
    # --- CLEANING STEP ---
    text = HTML_PATTERN.sub(' ', text) # Cleans <p>, <br>, etc.
    text = ENTITY_PATTERN.sub(' ', text)
    text = DASH_PATTERN.sub(' ', text)
    
    text = contractions.fix(text)
    
    # Clean other symbols but KEEP the quotes we just added
    text = CLEAN_PATTERN.sub('', text)
    
    return WHITESPACE_PATTERN.sub(' ', text).strip()

### Data Cleaning Comparison (Pandas vs Polars)
Let's compare the cleaning logic (filling nulls, string replacements) on the big dataset.

In [11]:
print("Cleaning Big Data with Pandas...")
subset_clean = df 
%time subset_clean['genres'] = subset_clean['genres'].astype(str).replace("[]", "Entertainment").str.replace(r"[\[\]\']", "", regex=True)
del subset_clean

Cleaning Big Data with Pandas...
CPU times: total: 1.69 s
Wall time: 1.8 s


In [ ]:
print("Cleaning Big Data with Polars...")
pl_df = pl.read_parquet("big_data.parquet")
%time pl_df = pl_df.with_columns(pl.col("genres").cast(pl.String).replace("[]", "Entertainment").str.replace_all(r"[\[\]\']", ""))
del pl_df

Cleaning Big Data with Polars...
CPU times: total: 484 ms
Wall time: 173 ms


### Comparing `text_process` on Big Dataset
We will now compare how long it takes to process the `summary` column across different libraries.

In [13]:
print("Running Standard Pandas Apply (on a subset of 500k rows for baseline)...")
subset = df.head(500000).copy()
%time _ = subset['summary'].apply(text_process)

Running Standard Pandas Apply (on a subset of 500k rows for baseline)...
CPU times: total: 26.8 s
Wall time: 27.1 s


In [14]:
print("Running Swifter (Pandas Parallelization)...")
# Swifter automatically decides between Dask and Pandas depending on which is faster
%time subset['processed_summary_swifter'] = subset['summary'].swifter.apply(text_process)

Running Swifter (Pandas Parallelization)...


Pandas Apply:   0%|          | 0/500000 [00:00<?, ?it/s]

CPU times: total: 27.3 s
Wall time: 27.9 s


In [4]:
import modin.pandas as mpd
print("Running Modin (Distributed Pandas on Ray/Dask)...")
# Convert to Modin DataFrame
m_df = mpd.read_parquet('big_data.parquet')
%time m_df['processed_summary'] = m_df['summary'].apply(text_process)

Running Modin (Distributed Pandas on Ray/Dask)...
CPU times: total: 15.6 ms
Wall time: 526 ms


### Conclusion on Processing
For massive datasets, Modin and Polars typically outperform standard Pandas by utilizing all available CPU cores. Swifter provides a nice middle ground by automatically choosing the best path.

In [5]:
%%time
m_df['processed_text'] = (
    "Title: " + m_df['name'].astype(str) + ". " +
    "Type: " + m_df['type'].astype(str) + ". " +
    "Language: " + m_df['language'].astype(str) + ". " +
    "Genres: " + m_df['genres'].astype(str) + ". " +
    "Premiered: " + m_df['premiered'].astype(str) + ". " +
    "Rating: " + m_df['rating'].astype(str) + ". " +
    "Description: " + m_df['processed_summary'].astype(str)
)


CPU times: total: 3.45 s
Wall time: 1min 47s


Seeing the speed on GPU for larger dataset

In [ ]:
embeddings = model.encode(
    m_df['processed_text'].tolist(), 
    batch_size=128, 
    show_progress_bar=True,
    convert_to_numpy=True # Keeps the output as a numpy array
)

# Add the embeddings to your dataframe
# We convert the numpy array to a list so it fits into a single column
df['embeddings'] = list(embeddings)

print(f"Embedding shape: {embeddings.shape}")